In [3]:
import os
import numpy as np

import qkeras
from qkeras import QActivation
from qkeras import QConv1D
from qkeras.quantizers import quantized_bits, quantized_relu, quantized_tanh

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger



resolution = 150
filters = 64
name =f"no_opt_{resolution}_{filters}_regular"
X_total = np.load(f"Data/X_Data_Bank_{resolution}.npy")
Y_total = np.load(f"Data/Y_Data_Bank_{resolution}.npy")
X_data = X_total[:1500,:,:]
y_data = Y_total[:1500,:,:]
TIME_STEPS = np.shape(X_data)[1]

VAL_SPLIT = 0.1

SAVE_DIR_opt = f'Output_drives/1Conv_checkpoints_running_{name}'
LOG_FILE_opt = f'Log_files/1Conv_3pulse_noise_tb23_{name}.csv'
MODEL_NAME_TEMPLATE_opt = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv_' + f'{name}.h5'
checkpoint_path_opt = os.path.join(SAVE_DIR_opt, MODEL_NAME_TEMPLATE_opt)

model = Sequential([
    Input(shape=(TIME_STEPS, 1)),
    Conv1D(filters, 3, padding='same', activation = "relu"),
    Conv1D(filters, 3, padding='same'),
    Conv1D(filters, 3, padding='same'),
    Conv1D(1, 1, padding='same'),
    Activation('sigmoid')
    ])

model.summary()


optimizer = Adam(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer) 


if not os.path.isdir(SAVE_DIR_opt):
    os.makedirs(SAVE_DIR_opt)

callbacks = [
    ModelCheckpoint(checkpoint_path_opt, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
    CSVLogger(LOG_FILE_opt, append=True, separator=','),
    EarlyStopping(monitor="val_loss", mode="min", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1) 
]


model.fit(
    X_data, y_data,
    epochs=300,                  
    shuffle=True,
    validation_split=VAL_SPLIT,
    callbacks=callbacks
)

if not os.path.isdir("hgdream_new"):
    os.makedirs("hgdream_new")
    
model.save(f"hgdream_new/quantized_model_{name}.h5")

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_4 (Conv1D)           (None, 80, 64)            256       
                                                                 
 conv1d_5 (Conv1D)           (None, 80, 64)            12352     
                                                                 
 conv1d_6 (Conv1D)           (None, 80, 64)            12352     
                                                                 
 conv1d_7 (Conv1D)           (None, 80, 1)             65        
                                                                 
 activation_1 (Activation)   (None, 80, 1)             0         
                                                                 
Total params: 25,025
Trainable params: 25,025
Non-trainable params: 0
_________________________________________________________________
Epoch 1/300
41/43 [===========================>..]

43/43 [==============================] - 0s 3ms/step - loss: 0.1132 - val_loss: 0.1137 - lr: 5.0000e-04
Epoch 26/300
22/43 [==============>...............] - ETA: 0s - loss: 0.1105
Epoch 26: val_loss did not improve from 0.11366
43/43 [==============================] - 0s 3ms/step - loss: 0.1129 - val_loss: 0.1141 - lr: 5.0000e-04
Epoch 27/300
23/43 [===============>..............] - ETA: 0s - loss: 0.1124
Epoch 27: val_loss did not improve from 0.11366
43/43 [==============================] - 0s 3ms/step - loss: 0.1133 - val_loss: 0.1139 - lr: 5.0000e-04
Epoch 28/300
23/43 [===============>..............] - ETA: 0s - loss: 0.1143
Epoch 28: val_loss improved from 0.11366 to 0.11365, saving model to 1Conv_checkpoints_running_no_opt_150_64_regular/1Conv_2pulse_noise.loss_0.11365.e028_deconv_no_opt_150_64_regular.h5
43/43 [==============================] - 0s 3ms/step - loss: 0.1132 - val_loss: 0.1136 - lr: 5.0000e-04
Epoch 29/300
23/43 [===============>..............] - ETA: 0s - loss: 0


Epoch 55: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.
43/43 [==============================] - 0s 3ms/step - loss: 0.1122 - val_loss: 0.1133 - lr: 3.1250e-05
Epoch 56/300
23/43 [===============>..............] - ETA: 0s - loss: 0.1153
Epoch 56: val_loss did not improve from 0.11329
43/43 [==============================] - 0s 3ms/step - loss: 0.1122 - val_loss: 0.1133 - lr: 1.5625e-05
Epoch 57/300
24/43 [===============>..............] - ETA: 0s - loss: 0.1145
Epoch 57: val_loss did not improve from 0.11329
43/43 [==============================] - 0s 3ms/step - loss: 0.1122 - val_loss: 0.1133 - lr: 1.5625e-05
Epoch 58/300
23/43 [===============>..............] - ETA: 0s - loss: 0.1133
Epoch 58: val_loss did not improve from 0.11329
43/43 [==============================] - 0s 3ms/step - loss: 0.1122 - val_loss: 0.1133 - lr: 1.5625e-05
Epoch 59/300
23/43 [===============>..............] - ETA: 0s - loss: 0.1114
Epoch 59: val_loss did not improve from 0.11329
43/43

In [1]:
import hls4ml
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)
print(co)
hls_config = hls4ml.utils.config_from_keras_model(
    qmodel,
    granularity='name',
    backend='Vitis'
)
hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    backend='Vitis',
    output_dir='model_final',
    part='xcu250-figd2104-2L-e',
    io_type='io_stream',
)
hls_model.compile()

2026-05-14 10:18:09.801218: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-14 10:18:09.838436: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Matplotlib created a temporary config/cache directory at /tmp/matplotlib-7spbo_5p because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support 

{'QInitializer': <class 'qkeras.qlayers.QInitializer'>, 'QDense': <class 'qkeras.qlayers.QDense'>, 'QConv1D': <class 'qkeras.qconvolutional.QConv1D'>, 'QConv2D': <class 'qkeras.qconvolutional.QConv2D'>, 'QConv2DTranspose': <class 'qkeras.qconvolutional.QConv2DTranspose'>, 'QSimpleRNNCell': <class 'qkeras.qrecurrent.QSimpleRNNCell'>, 'QSimpleRNN': <class 'qkeras.qrecurrent.QSimpleRNN'>, 'QLSTMCell': <class 'qkeras.qrecurrent.QLSTMCell'>, 'QLSTM': <class 'qkeras.qrecurrent.QLSTM'>, 'QGRUCell': <class 'qkeras.qrecurrent.QGRUCell'>, 'QGRU': <class 'qkeras.qrecurrent.QGRU'>, 'QBidirectional': <class 'qkeras.qrecurrent.QBidirectional'>, 'QDepthwiseConv2D': <class 'qkeras.qconvolutional.QDepthwiseConv2D'>, 'QSeparableConv1D': <class 'qkeras.qconvolutional.QSeparableConv1D'>, 'QSeparableConv2D': <class 'qkeras.qconvolutional.QSeparableConv2D'>, 'QActivation': <class 'qkeras.qlayers.QActivation'>, 'QAdaptiveActivation': <class 'qkeras.qlayers.QAdaptiveActivation'>, 'QBatchNormalization': <class

NameError: name 'qmodel' is not defined